# step3 CCA

In [ ]:
library(ggplot2) 
library(dplyr)
library(tidyverse)
library(tidyverse)

set.seed(1)
options(stringsAsFactors=F)
options(scipen=100)

In [ ]:
# ---- Set the following paths to match your environment ----
data_path  = "/path/to/data"            # directory for data
count_file = "/path/to/counts.txt"      # RNA-seq count file
# -----------------------------------------------------------

out_f=paste0(data_path,"/gene_filter")
dir.create(out_f, recursive = T)


TF_df=read.table(paste0(out_f, "/TF_matrix.txt"), header=1,row.names=1)
transcriptome_df=read.table(paste0(out_f,  "/transcriptome_matrix.txt"), header=1, row.names=1)
i=10000

transcriptome_df_2   = transcriptome_df %>% mutate(var=apply(.,1,var))
transcriptome_df_3   = transcriptome_df_2[order(transcriptome_df_2$var,decreasing=T),] %>% .[1:as.numeric(i),] %>% dplyr::select(-var)

TF_df_2   = TF_df %>% mutate(var=apply(.,1,var))
TF_df_3   = TF_df_2[order(TF_df_2$var,decreasing=T),] %>% .[1:as.numeric(i),] %>% dplyr::select(-var)

if (nrow(transcriptome_df_3)>10000){
    highly_variable_genes=intersect(rownames(TF_df_3), rownames(transcriptome_df_3))
}else{
    highly_variable_genes=intersect(rownames(TF_df_3), rownames(transcriptome_df))
}
    
gene_num=length(highly_variable_genes)
TF_df_highly=TF_df[highly_variable_genes,]
transcriptome_df_highly=transcriptome_df[highly_variable_genes,]

sum=apply(transcriptome_df_highly,1,sum)
if (min(sum)==0){
    print("Certain genes were not detected in any cells.")
}

sum2=apply(transcriptome_df_highly,2,sum)
if (min(sum)==0){
    print("Certain cells had no measurable gene expression.")
}

TF_max=apply(TF_df_highly,2,max)
TF_min=apply(TF_df_highly,2,min)

TF_allzero=TF_max[TF_max==0] 
TF_allzero_names=names(TF_allzero)
TF_df_highly_nonzero=TF_df_highly %>% dplyr::select(-TF_allzero_names)


X <- as.matrix(TF_df_highly_nonzero)
Y <- as.matrix(transcriptome_df_highly)



scaled_X= apply(X,1,function(x){ (x - mean(x))/sd(x) }) 
scaled_Y= apply(Y,1,function(x){ (x - mean(x))/sd(x) })
scaled_X2= apply(scaled_X,1,function(x){ (x - mean(x))/sd(x) }) 
scaled_Y2= apply(scaled_Y,1,function(x){ (x - mean(x))/sd(x) })
scaled_Y2_t=t(scaled_Y2)
scaled_X2_t=t(scaled_X2)
NA_colnames <-as.data.frame(scaled_X2_t) %>%
  select_if(function(x) any(is.na(x))) %>%
  colnames()

length(NA_colnames) 

scaled_X3_t=as.data.frame(scaled_X2_t) %>% dplyr::select(-all_of(NA_colnames)) %>% as.matrix()

scaled_Y3=scaled_Y2[colnames(scaled_X3_t),]
scaled_Y3_t=t(scaled_Y3)

XtY=scaled_X3_t %*% scaled_Y3

# SVD
svd_res=svd(XtY)

u=svd_res$u[,1:10]
X_u_df=as.data.frame(u)

colnames(X_u_df)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

X_u_df$id=colnames(TF_df_highly_nonzero)

X_u_df2=X_u_df

v=svd_res$v[,1:10]
Y_v_df=as.data.frame(v)

colnames(Y_v_df)=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

Y_v_df$id=colnames(transcriptome_df_highly)



out_f3=paste0(out_f,"/raw_CC")
dir.create(out_f3, recursive = T)

                        
CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

for (CC in CC_list){
    CC_df=X_u_df[,c("id",CC)]
    write.table(CC_df, paste0(out_f3, "/",  CC, "_CCscore.txt"),row.names=FALSE,col.names=FALSE, sep="\t", append=F, quote=F)
}

out_f4=paste0(out_f,"/transcriptome_raw_CC")
dir.create(out_f4, recursive = T)

                        
CC_list=c("CC1_rotation", "CC2_rotation", "CC3_rotation", "CC4_rotation", "CC5_rotation", "CC6_rotation", "CC7_rotation", "CC8_rotation", "CC9_rotation", "CC10_rotation")

for (CC in CC_list){
    CC_df=Y_v_df[,c("id",CC)]
    write.table(CC_df, paste0(out_f4, "/",  CC, "_CCscore.txt"),row.names=FALSE,col.names=FALSE, sep="\t", append=F, quote=F)
}

all_svd_vec=svd_res$d
rownames(Y_v_df)=Y_v_df$id
Y_v_df_Cor=Y_v_df %>% dplyr::select(-id)

            

## output parameters

In [ ]:
#output parameters

merge_df=data.frame(matrix(rep(0, 10), ncol=10))[0,]
valiance_list=data.frame(matrix(rep(0, 10), ncol=10))
colnames(valiance_list)=c(paste0("CC", seq(10)))
for (CC_n in seq(10)){
    for (i in seq(ncol(scaled_Y3_t))){
        valiance_list[1,CC_n]=valiance_list[1,CC_n]+((cor(scaled_Y3_t[,i], Y_v_df_Cor[,CC_n]))**2)
    }
}
val_df=valiance_list/ncol(scaled_Y3_t)
rownames(val_df)=c("transcriptome_variance_explained_by_CC")

X_u_df_val=X_u_df %>% dplyr::select(-id)
valiance_list_x=data.frame(matrix(rep(0, 10), ncol=10))
colnames(valiance_list_x)=c(paste0("CC", seq(10)))
for (k in seq(10)){
    for (i in seq(ncol(scaled_X3_t))){
        valiance_list_x[1,k]=valiance_list_x[1,k]+((cor(scaled_X3_t[,i], X_u_df_val[,k]))**2)
    }
}
val_df_x=valiance_list_x/ncol(scaled_X3_t)
rownames(val_df_x)=c("TF_variance_explained_by_CC")
merge_df=rbind(val_df, val_df_x)

all_svd_vec=svd_res$d
sv_df=data.frame(matrix(rep(0, 10), ncol=10))
colnames(sv_df)=c(paste0("CC", seq(10)))
for (i in seq(10)){
    sv_df[1,i]=(all_svd_vec[i])/sum(all_svd_vec)
}
rownames(sv_df)=c("sv")
merge_df=rbind(merge_df, sv_df)

SCF_df=data.frame(matrix(rep(0, 10), ncol=10))
colnames(SCF_df)=c(paste0("CC", seq(10)))
for (i in seq(10)){
    SCF_df[1,i]=(svd_res$d[i])^2/sum(all_svd_vec^2)
}
rownames(SCF_df)=c("SCF")
merge_df=rbind(merge_df, SCF_df)

rownames(X_u_df)=X_u_df$id
X_u_df_Cor=X_u_df %>% dplyr::select(-c("id")) %>% t() %>% as.matrix()
scaled_X4=scaled_X3_t %>% as.matrix()
a=X_u_df_Cor　%*% scaled_X4
Y_v_df_Cormat=Y_v_df %>% dplyr::select(-c("id")) %>% as.matrix()
b=scaled_Y3 %*% Y_v_df_Cormat
cor_df=data.frame(matrix(rep(0, 10), ncol=10))
colnames(cor_df)=c(paste0("CC", seq(10)))
for (i in seq(10)){
    cor_df[1,i]=cor(a[i,], b[,i])
}
rownames(cor_df)=c("canonical_correlation")
para_df=rbind(merge_df, cor_df)



write.table(data.frame(parameter_name=rownames(para_df), para_df), paste0(out_f, "/CCA_parameter.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

## graphs

In [ ]:
# ---- Set the following paths to match your environment ----
meta_file  = "/path/to/metadata.txt"    # sample metadata, including columns of "cell id" for cell id and "cluster" for cluster id
# ------------------------------------------------------------
cluster_df = read.table(meta_file, header=1)

out_f2=paste0(out_f, "/plots")
dir.create(out_f2)

cluster_df$cell_id2=gsub("-", "\\.",cluster_df$cell_id)
Y_v_df$cell_id2=rownames(Y_v_df)
Y_v_df2=left_join(Y_v_df, cluster_df, by="cell_id2")
Y_v_df2$cluster_id=as.character(Y_v_df2$cluster_id)
g <- ggplot(Y_v_df2, aes(CC1_rotation, CC2_rotation, fill=cluster_id))
g + geom_point(shape = 21, size=1, alpha=0.7)+xlab(paste0("CC1_rotation SCF=", round((svd_res$d[1])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC2_rotation SCF=", round((svd_res$d[2])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))
ggsave(paste0(out_f2, "/CCA_rotation_CC1_CC2_sc_clus.pdf"), width=5, height=4)

g <- ggplot(Y_v_df2, aes(CC3_rotation, CC4_rotation, fill=cluster_id))
g + geom_point(shape = 21, size=1, alpha=0.7)+xlab(paste0("CC3_rotation SCF=", round((svd_res$d[3])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC4_rotation SCF=", round((svd_res$d[4])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))
ggsave(paste0(out_f2, "/CCA_rotation_CC3_CC4_sc_clus.pdf"), width=5, height=4)

g <- ggplot(Y_v_df2, aes(CC5_rotation, CC6_rotation, fill=cluster_id))
g + geom_point(shape = 21, size=1, alpha=0.7)+xlab(paste0("CC5_rotation SCF=", round((svd_res$d[5])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC6_rotation SCF=", round((svd_res$d[6])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))
ggsave(paste0(out_f2, "/CCA_rotation_CC5_CC6_sc_clus.pdf"), width=5, height=4)

g <- ggplot(Y_v_df2, aes(CC7_rotation, CC8_rotation, fill=cluster_id))
g + geom_point(shape = 21, size=1, alpha=0.7)+xlab(paste0("CC7_rotation SCF=", round((svd_res$d[7])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC8_rotation SCF=", round((svd_res$d[8])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))
ggsave(paste0(out_f2, "/CCA_rotation_CC7_CC8_sc_clus.pdf"), width=5, height=4)

g <- ggplot(Y_v_df2, aes(CC9_rotation, CC10_rotation, fill=cluster_id))
g + geom_point(shape = 21, size=1, alpha=0.7)+xlab(paste0("CC9_rotation SCF=", round((svd_res$d[9])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC10_rotation SCF=", round((svd_res$d[10])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))
ggsave(paste0(out_f2, "/CCA_rotation_CC9_CC10_sc_clus.pdf"), width=5, height=4)

In [ ]:
Y_woNA=Y[colnames(scaled_X3_t),]
Y_t=t(Y_woNA)

PCA_result=prcomp(Y_t, scale=TRUE)
summ=summary(PCA_result)


PCA_result_df=as.data.frame(PCA_result$x)



PCA_result_df$id=rownames(PCA_result_df)
PCA_result_df$cell_id2=rownames(PCA_result_df)
PCA_result_df2=left_join(PCA_result_df, cluster_df, by="cell_id2")



g <- ggplot(PCA_result_df2, aes(PC1, PC2, shape=cluster_id))
g + geom_point(size=1, alpha=0.7)+xlab(paste0("PC1 Variance explained=", round(summ$importance[2,1],2)*100, "%"))+ylab(paste0("PC2 Variance explained=", round(summ$importance[2,2]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))+ scale_shape_manual(values=seq(1,16))
ggsave(paste0(out_f2, "/sc_PCA_PC1_PC2_clus_shape.pdf"), width=5, height=4)

g <- ggplot(PCA_result_df2, aes(PC3, PC4, shape=cluster_id))
g + geom_point(size=1, alpha=0.7)+xlab(paste0("PC3 Variance explained=", round(summ$importance[2,3],2)*100, "%"))+ylab(paste0("PC4 Variance explained=", round(summ$importance[2,4]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))+ scale_shape_manual(values=seq(1,16))
ggsave(paste0(out_f2, "/sc_PCA_PC3_PC4_clus_shape.pdf"), width=5, height=4)


In [ ]:

out_f2=paste0(out_f, "/plots")
dir.create(out_f2, recursive=T)

g <- ggplot(X_u_df, aes(CC1_rotation, CC2_rotation))
g + geom_point(shape = 21, size=3, alpha=0.7)+xlab(paste0("CC1_rotation SCF=", round((svd_res$d[1])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC2_rotation SCF=", round((svd_res$d[2])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+geom_text(aes(label=id), size=3)+
theme_bw()
ggsave(paste0(out_f2, "/CCA_rotation_CC1_CC2.pdf"), width=30, height=30)

g <- ggplot(X_u_df, aes(CC3_rotation, CC4_rotation))
g + geom_point(shape = 21, size=3, alpha=0.7)+xlab(paste0("CC3_rotation SCF=", round((svd_res$d[3])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC4_rotation SCF=", round((svd_res$d[4])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+geom_text(aes(label=id), size=3)+
theme_bw()
ggsave(paste0(out_f2, "/CCA_rotation_CC3_CC4.pdf"), width=30, height=30)

g <- ggplot(X_u_df, aes(CC5_rotation, CC6_rotation))
g + geom_point(shape = 21, size=3, alpha=0.7)+xlab(paste0("CC5_rotation  SCF=", round((svd_res$d[5])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC6_rotation SCF=", round((svd_res$d[6])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+geom_text(aes(label=id), size=3)+
theme_bw()
ggsave(paste0(out_f2, "/CCA_rotation_CC5_CC6.pdf"), width=30, height=30)
    
g <- ggplot(X_u_df, aes(CC7_rotation, CC8_rotation))
g + geom_point(shape = 21, size=3, alpha=0.7)+xlab(paste0("CC7_rotation SCF=", round((svd_res$d[7])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC8_rotation SCF=", round((svd_res$d[8])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+geom_text(aes(label=id), size=3)+
theme_bw()
ggsave(paste0(out_f2, "/CCA_rotation_CC7_CC8.pdf"), width=30, height=30)

g <- ggplot(X_u_df, aes(CC9_rotation, CC10_rotation))
g + geom_point(shape = 21, size=3, alpha=0.7)+xlab(paste0("CC9_rotation SCF=", round((svd_res$d[9])^2/sum(all_svd_vec^2),2)*100, "%"))+ylab(paste0("CC10_rotation SCF=", round((svd_res$d[10])^2/sum(all_svd_vec^2),2)*100, "%"))+ theme(text = element_text(size = 24))+geom_text(aes(label=id), size=3)+
theme_bw()
ggsave(paste0(out_f2, "/CCA_rotation_CC9_CC10.pdf"), width=30, height=30)
    





X_woNA=X[colnames(scaled_X3_t), ]
X_t=t(X_woNA)

PCA_result=prcomp(X_t, scale=TRUE)
summ=summary(PCA_result)
  

PCA_result_df=as.data.frame(PCA_result$x)

PCA_result_df$id=rownames(PCA_result_df)




g <- ggplot(PCA_result_df, aes(PC1, PC2))+geom_text(aes(label=id), size=3)
g + geom_point(shape = 21, size=3, alpha=0.7) +xlab(paste0("PC1 Variance explained=", round(summ$importance[2,1],2)*100, "%"))+ylab(paste0("PC2 Variance explained=", round(summ$importance[2,2]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()
ggsave(paste0(out_f2, "/TF_PCA_PC1_PC2.pdf"), width=30, height=30)

g <- ggplot(PCA_result_df, aes(PC3, PC4))+geom_text(aes(label=id), size=3)
g + geom_point(shape = 21, size=3, alpha=0.7) +xlab(paste0("PC3 Variance explained=", round(summ$importance[2,3],2)*100, "%"))+ylab(paste0("PC4 Variance explained=", round(summ$importance[2,4]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()
ggsave(paste0(out_f2, "/TF_PCA_PC3_PC4.pdf"), width=30, height=30)

g <- ggplot(PCA_result_df, aes(PC5, PC6))+geom_text(aes(label=id), size=3)
g + geom_point(shape = 21, size=3, alpha=0.7) +xlab(paste0("PC5 Variance explained=", round(summ$importance[2,5],2)*100, "%"))+ylab(paste0("PC6 Variance explained=", round(summ$importance[2,6]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()
ggsave(paste0(out_f2, "/TF_PCA_PC5_PC6.pdf"), width=30, height=30)

g <- ggplot(PCA_result_df, aes(PC7, PC8))+geom_text(aes(label=id), size=3)
g + geom_point(shape = 21, size=3, alpha=0.7) +xlab(paste0("PC7 Variance explained=", round(summ$importance[2,7],2)*100, "%"))+ylab(paste0("PC8 Variance explained=", round(summ$importance[2,8]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()
ggsave(paste0(out_f2, "/TF_PCA_PC7_PC8.pdf"), width=30, height=30)

g <- ggplot(PCA_result_df, aes(PC9, PC10))+geom_text(aes(label=id), size=3)
g + geom_point(shape = 21, size=3, alpha=0.7) +xlab(paste0("PC9 Variance explained=", round(summ$importance[2,9],2)*100, "%"))+ylab(paste0("PC10 Variance explained=", round(summ$importance[2,10]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()
ggsave(paste0(out_f2, "/TF_PCA_PC9_PC10.pdf"), width=30, height=30)





In [ ]:
Y_woNA=Y[colnames(scaled_X3_t),]
Y_t=t(Y_woNA)

PCA_result=prcomp(Y_t, scale=TRUE)
summ=summary(PCA_result)


PCA_result_df=as.data.frame(PCA_result$x)



PCA_result_df$id=rownames(PCA_result_df)
PCA_result_df$cell_id2=rownames(PCA_result_df)
PCA_result_df2=left_join(PCA_result_df, cluster_df, by="cell_id2")


g <- ggplot(PCA_result_df2, aes(PC1, PC2,fill=cluster_id))
g + geom_point(shape = 21, size=1, alpha=0.7)+xlab(paste0("PC1 Variance explained=", round(summ$importance[2,1],2)*100, "%"))+ylab(paste0("PC2 Variance explained=", round(summ$importance[2,2]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))
ggsave(paste0(out_f2, "/sc_PCA_PC1_PC2.pdf"), width=5, height=4)

g <- ggplot(PCA_result_df2, aes(PC3, PC4,fill=cluster_id))
g + geom_point(shape = 21, size=1, alpha=0.7)+xlab(paste0("PC3 Variance explained=", round(summ$importance[2,3],2)*100, "%"))+ylab(paste0("PC4 Variance explained=", round(summ$importance[2,4]*100,2), "%"))+ theme(text = element_text(size = 24))+
theme_bw()+guides(fill=guide_legend(title=NULL))
ggsave(paste0(out_f2, "/sc_PCA_PC3_PC4.pdf"), width=5, height=4)




sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Red Hat Enterprise Linux 8.8 (Ootpa)

Matrix products: default
BLAS/LAPACK: /rshare1/ZETTAI_path_WA_slash_home_KARA/home/imgtaka/miniconda3/envs/renv/lib/libopenblasp-r0.3.26.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=ja_JP.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=ja_JP.UTF-8        LC_COLLATE=ja_JP.UTF-8    
 [5] LC_MONETARY=ja_JP.UTF-8    LC_MESSAGES=ja_JP.UTF-8   
 [7] LC_PAPER=ja_JP.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=ja_JP.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Tokyo
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] digest_0.6.35   IRdisplay_1.1   utf8_1.2.4      base64enc_0.1-3
 [5] fastmap_1.1.1   glue_1.7.0      htmltools_0.5.8 repr_1.1.7     
 [9] lifecycle_1.0.4 cli_3.6.2       fansi_1.0.6     vctrs_0.6.5    
[13] pbdZMQ_0.3-11   compiler_4.3.3  tools_4.3.3     evaluate_0.23  
[17] pillar_1.9.0    crayon_1.5.2    rlang_1.1.3     jsonlite_1.8.8 
[21] IRkernel_1.3.2  uuid_1.2-0  